### ENET FOR IMAGES

In [1]:
# import the necessary packages
import numpy as np
import argparse
import imutils
import time
import cv2
import os

In [2]:
# read the sample input image, resize the image, and construct a blob from the sample image
start = time.time()
SET_WIDTH = int(600)

normalize_image = 1 / 255.0
resize_image_shape = (1024, 512)

sample_img = cv2.imread('./images/example_02.jpg')
sample_img = imutils.resize(sample_img, width=SET_WIDTH)

blob_img = cv2.dnn.blobFromImage(sample_img, normalize_image, resize_image_shape, 0, swapRB=True, crop=False)

In [3]:
blob_img

array([[[[0.08627451, 0.09019608, 0.09411765, ..., 0.36078432,
          0.2901961 , 0.24705882],
         [0.09411765, 0.09803922, 0.10196079, ..., 0.37254903,
          0.32941177, 0.30588236],
         [0.10588235, 0.10980392, 0.11372549, ..., 0.3882353 ,
          0.39215687, 0.39215687],
         ...,
         [0.15686275, 0.15686275, 0.15686275, ..., 0.10588235,
          0.10588235, 0.10588235],
         [0.15686275, 0.15686275, 0.15686275, ..., 0.10980392,
          0.10980392, 0.10980392],
         [0.16078432, 0.16078432, 0.16078432, ..., 0.11372549,
          0.11372549, 0.11372549]],

        [[0.10588235, 0.10980392, 0.11372549, ..., 0.39607844,
          0.3254902 , 0.2784314 ],
         [0.11764706, 0.11764706, 0.12156863, ..., 0.40392157,
          0.3647059 , 0.3372549 ],
         [0.13333334, 0.13333334, 0.13725491, ..., 0.42352942,
          0.42352942, 0.42352942],
         ...,
         [0.18431373, 0.18431373, 0.18431373, ..., 0.13333334,
          0.13333334, 0.1

In [4]:
# load the serialized ENET model from disk
print("[INFO] loading model...")
cv_enet_model = cv2.dnn.readNet('./enet-cityscapes/enet-model.net')

[INFO] loading model...


In [5]:
# perform a forward pass using the segmentation model
cv_enet_model.setInput(blob_img)

cv_enet_model_output = cv_enet_model.forward()

In [6]:
# load the class name labels
label_values = open('./enet-cityscapes/enet-classes.txt').read().strip().split("\n")

In [7]:
# infer the shape of the total number of classes, along with the spatial dimensions of the mask image
IMG_OUTPUT_SHAPE_START = 1
IMG_OUTPUT_SHAPE_END = 4
(classes_num, h, w) = cv_enet_model_output.shape[IMG_OUTPUT_SHAPE_START:IMG_OUTPUT_SHAPE_END]

In [8]:
# the output class ID map will be numclasses * height * width in size;
# therefore, we take argmax to find the class label with the highest probability for each and every (x, y) coordinate
class_map = np.argmax(cv_enet_model_output[0], axis=0)

In [9]:
# if we have colors file, we can load it from disk; otherwise we need to randomly generate RGB colors for each class.
# a list of colors is initialized to represent each class
import os
if os.path.isfile('./enet-cityscapes/enet-colors.txt'):
    CV_ENET_SHAPE_IMG_COLORS = open('./enet-cityscapes/enet-colors.txt').read().strip().split("\n")
    CV_ENET_SHAPE_IMG_COLORS = [np.array(c.split(",")).astype("int") for c in CV_ENET_SHAPE_IMG_COLORS]
    CV_ENET_SHAPE_IMG_COLORS = np.array(CV_ENET_SHAPE_IMG_COLORS, dtype="uint8")

else:
   
    np.random.seed(42)
    CV_ENET_SHAPE_IMG_COLORS = np.random.randint(0, 255, size=(len(label_values) - 1, 3),
                               dtype="uint8")
    CV_ENET_SHAPE_IMG_COLORS = np.vstack([[0, 0, 0], CV_ENET_SHAPE_IMG_COLORS]).astype("uint8")

In [10]:
# map each class id with the given class id 
mask_class_map = CV_ENET_SHAPE_IMG_COLORS[class_map]

In [11]:
# we will resize the mask and class map in such way that its dimensions match the original size of the input image
mask_class_map = cv2.resize(mask_class_map, (sample_img.shape[1], sample_img.shape[0]),
                  interpolation=cv2.INTER_NEAREST)

class_map = cv2.resize(mask_class_map, (sample_img.shape[1], sample_img.shape[0]),
                      interpolation=cv2.INTER_NEAREST)

In [12]:
# we will get a weighted combination of the input image and the mask to form a visualized output.
# here, mask to image means filtering the image.
# the sum of the weight in the convolution mask effects the overall intensity of the resulting image.
# in our case, it is the sum of 0.4 and 0.6 that is 1.
# pixels with negative values may be generated using masks with negative weights.
cv_enet_model_output = ((0.4 * sample_img) + (0.6 * mask_class_map)).astype("uint8")

In [13]:
# then, we initiliaze the legend's visualization
my_legend = np.zeros(((len(label_values) * 25) + 25, 300, 3), dtype="uint8")

In [14]:
# it will loop over the class names and colors, thereby drawing the class name and color on the legend
for (i, (class_name, img_color)) in enumerate(zip(label_values, CV_ENET_SHAPE_IMG_COLORS)):
    # draw the class name + color on the legend
    color_info = [int(color) for color in img_color]
    cv2.putText(my_legend, class_name, (5, (i * 25) + 17),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    cv2.rectangle(my_legend, (100, (i * 25)), (300, (i * 25) + 25),
                  tuple(color_info), -1)

In [15]:
# we can now show the input and output images.
cv2.imshow("My_Legend", my_legend)
cv2.imshow("Img_Input", sample_img)
cv2.imshow("CV_Model_Output", cv_enet_model_output)
cv2.waitKey(0)
cv2.destroyAllWindows()

end = time.time()

# after that, we can show the amount of time the inference took
print("[INFO] inference took {:.4f} seconds".format(end - start))


[INFO] inference took 19.2101 seconds


### ENET FOR VIDEOS

In [16]:
import os
import time
import cv2
import imutils
import numpy as np


DEFAULT_FRAME = 1
SET_WIDTH = 600

In [17]:
# load the class label names
class_labels = open('./enet-cityscapes/enet-classes.txt').read().strip().split("\n")

In [18]:
# load the colors from file or generate them randomly.
if os.path.isfile('./enet-cityscapes/enet-colors.txt'):
    CV_ENET_SHAPE_IMG_COLORS = open('./enet-cityscapes/enet-colors.txt').read().strip().split("\n")
    CV_ENET_SHAPE_IMG_COLORS = [np.array(c.split(",")).astype("int") for c in CV_ENET_SHAPE_IMG_COLORS]
    CV_ENET_SHAPE_IMG_COLORS = np.array(CV_ENET_SHAPE_IMG_COLORS, dtype="uint8")

else:
    
    np.random.seed(42)
    CV_ENET_SHAPE_IMG_COLORS = np.random.randint(0, 255, size=(len(class_labels) - 1, 3),
                               dtype="uint8")
    CV_ENET_SHAPE_IMG_COLORS = np.vstack([[0, 0, 0], CV_ENET_SHAPE_IMG_COLORS]).astype("uint8")

In [19]:
# load the model
print("[INFO] loading model...")
cv_enet_model = cv2.dnn.readNet('./enet-cityscapes/enet-model.net')

[INFO] loading model...


In [20]:
# initiliaze the video stream so that we can output the video file
sv = cv2.VideoCapture('.//video//video.mp4')
sample_video_writer = None

In [21]:
print(sv)

< cv2.VideoCapture 0000022E4C989490>


In [22]:
prop = cv2.cv.CV_CAP_PROP_FRAME_COUNT if imutils.is_cv2() \
        else cv2.CAP_PROP_FRAME_COUNT

In [23]:
total = int(sv.get(prop))

In [24]:
total

1007

In [27]:
# try to determine the total number of frames in the video file
try:
    prop = cv2.cv.CV_CAP_PROP_FRAME_COUNT if imutils.is_cv2() \
        else cv2.CAP_PROP_FRAME_COUNT
    total = int(sv.get(prop))
    print("[INFO] {} total frames in video".format(total))

# an error occurred while trying to determine the total
# number of frames in the video file
except:
    print("[INFO] could not determine # of frames in video")
    total = -1
#sample_video
# loop over frames from the video file stream
while True:
    # read the next frame from the file
    (grabbed, frame) = sv.read()

    # if the frame was not grabbed, then we have reached the end
    # of the stream
    if not grabbed:
        break

    # construct a blob from the frame and perform a forward pass
    # using the segmentation model
    start = time.time()
    normalize_image = 1 / 255.0
    resize_image_shape = (1024, 512)
    frame = imutils.resize(frame, width=SET_WIDTH)
    blob_img = cv2.dnn.blobFromImage(frame, normalize_image, resize_image_shape, 0, 
                                     swapRB=True, crop=False)
    cv_enet_model.setInput(blob_img)
    start = time.time()
    cv_enet_model_output = cv_enet_model.forward()
    end = time.time()

    # infer the total number of classes along with the spatial
    # dimensions of the mask image via the shape of the output array
    (classes_num, height, width) = cv_enet_model_output.shape[1:4]

    # our output class ID map will be num_classes x height x width in
    # size, so we take the argmax to find the class label with the
    # largest probability for each and every (x, y)-coordinate in the
    # image
    classMap = np.argmax(cv_enet_model_output[0], axis=0)

    # given the class ID map, we can map each of the class IDs to its
    # corresponding color
    
    mask_class_map = CV_ENET_SHAPE_IMG_COLORS[classMap]

    # resize the mask such that its dimensions match the original size
    # of the input frame
    
    
    mask_class_map = cv2.resize(mask_class_map, (frame.shape[1], frame.shape[0]),
                      interpolation=cv2.INTER_NEAREST)

    # perform a weighted combination of the input frame with the mask
    # to form an output visualization
    
    
    cv_enet_model_output = ((0.3 * frame) + (0.7 * mask_class_map)).astype("uint8")

    # check if the video writer is None
    if sample_video_writer is None:
        print("sample_video_writer is None")
        # initialize our video writer
        fourcc_obj = cv2.VideoWriter_fourcc(*"MJPG")

        sample_video_writer = cv2.VideoWriter('./output/output_toronoto.avi', fourcc_obj, 30,
                                 (cv_enet_model_output.shape[1], cv_enet_model_output.shape[0]), True)

        # some information on processing single frame
        if total > 0:
            
            execution_time = (end - start)
            print("[INFO] single video_frame took {:.4f} seconds".format(execution_time))

            print("[INFO] estimated total_time time: {:.4f}".format(
                execution_time * total))

    # write the output frame to disk
    
    sample_video_writer.write(cv_enet_model_output)

    # check to see if we should display the output frame to our screen
    if DEFAULT_FRAME > 0:
        cv2.imshow("Video Frame", cv_enet_model_output)
        
        key = cv2.waitKey(1) & 0xFF
        
        if key == ord("q"):
            break

print("[INFO] cleaning up...")
sample_video_writer.release()
sv.release()

[INFO] 1007 total frames in video
sample_video_writer is None
[INFO] single video_frame took 0.2389 seconds
[INFO] estimated total_time time: 240.5430
[INFO] cleaning up...
